# Smart Traffic Light Balanced Traffic-Police DQN Training (Kaggle)

This notebook trains a traffic-police-style controller from the `dev-truong` branch of `truongNgn/smart-traffic-light-system`.

The action space matches a real traffic officer at a Vietnamese four-way intersection:

- `Action 0 = EAST_WEST`: both opposite directions on the East-West road get green together.
- `Action 1 = NORTH_SOUTH`: both opposite directions on the North-South road get green together.

This version focuses on a more balanced deployment model. It patches the reward on Kaggle so the agent is not rewarded for clearing one busy road while starving the other road, then trains on normal, balanced-heavy, imbalanced peak, and a time-blocked mixed scenario. The final cell creates `dqn_eval_best.pt`, selected by weighted evaluation against fixed-time across all scenarios.


## 1. Install SUMO and clone the repo

In [ ]:
!pip install -q eclipse-sumo traci sumolib libsumo

In [ ]:
!git clone --branch dev-truong --single-branch https://github.com/truongNgn/smart-traffic-light-system.git
%cd smart-traffic-light-system


In [ ]:
# torch is already preinstalled on Kaggle's GPU image - don't reinstall it
!pip install -q pydantic pydantic-settings structlog gymnasium numpy tqdm

## 1.1. Verify 2-phase traffic-police logic

`dev-truong` is the default branch for this project. This cell verifies that Kaggle cloned a commit with the improved 2-phase action space before spending time training.


In [ ]:
# Verify the cloned dev-truong code uses the 2-phase action space and adaptive starvation guard.
from common.constants import DQN_OUTPUT_SIZE, NUM_ACTIONS, PHASE_DIRECTIONS, PhaseAction
from rl.env.traffic_env import SumoTrafficEnv
from rl.train.config import TrainingConfig

cfg = TrainingConfig()
assert NUM_ACTIONS == 2, f"Expected 2 phase actions, got {NUM_ACTIONS}"
assert DQN_OUTPUT_SIZE == 2, f"Expected DQN output size 2, got {DQN_OUTPUT_SIZE}"
assert set(PhaseAction) == {PhaseAction.EAST_WEST, PhaseAction.NORTH_SOUTH}
assert len(PHASE_DIRECTIONS[PhaseAction.EAST_WEST]) == 2
assert len(PHASE_DIRECTIONS[PhaseAction.NORTH_SOUTH]) == 2
for name in [
    "initial_phase",
    "max_red_time_s",
    "soft_red_time_s",
    "hard_red_time_s",
    "starving_queue_threshold",
    "starving_wait_time_s",
]:
    assert name in SumoTrafficEnv.__init__.__code__.co_varnames, name
for name in [
    "max_red_time_s",
    "soft_red_time_s",
    "hard_red_time_s",
    "starving_queue_threshold",
    "starving_wait_time_s",
]:
    assert hasattr(cfg, name), name
print("Verified: 2-phase actions + adaptive red-time starvation guard are available.")


In [ ]:
import os
import sumo

# eclipse-sumo bundles its own binaries + tools/ under the installed
# package directory - point SUMO_HOME there instead of a system path.
os.environ["SUMO_HOME"] = os.path.dirname(sumo.__file__)
print("SUMO_HOME =", os.environ["SUMO_HOME"])

!sumo --version

If `sumo --version` fails to print a version here, the `eclipse-sumo` wheel most likely didn't ship a binary for this exact platform. Fall back to the apt-get route as a last resort (slower, and known to segfault on some Kaggle images - see the note in cell 1):

```bash
!apt-get update -qq && apt-get install -y -qq sumo sumo-tools sumo-doc
```
```python
import os
os.environ["SUMO_HOME"] = "/usr/share/sumo"
```

## 2. Confirm GPU is visible to PyTorch, and libsumo is importable

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

import libsumo
print("libsumo importable OK - training will use the fast in-process backend")

## 3. Build the SUMO network and generate mixed demand scenarios

The last model improved peak imbalance but regressed on normal and balanced-heavy traffic. This notebook generates five scenarios:

- `normal`: original demand.
- `heavy_2x`: all approaches doubled.
- `imbalanced_ew3x`: East-West peak road, North-South normal.
- `imbalanced_ns3x`: North-South peak road, East-West normal.
- `mixed_balanced_peak`: one long episode with normal, heavy, EW peak, and NS peak blocks.


In [ ]:
!python -m simulation.net.build_net
!python -m simulation.net.generate_routes --duration 1800 --seed 42 --out simulation/net/intersection.rou.xml

from copy import deepcopy
from pathlib import Path
import xml.etree.ElementTree as ET


def flow_origin(flow):
    route = flow.get("route", "")
    # route_N_to_S -> N. Falling back to the id keeps compatibility with
    # older generated route files.
    if route.startswith("route_"):
        return route.split("_")[1]
    parts = flow.get("id", "").split("_")
    return parts[-2] if len(parts) >= 2 else ""


def write_sumocfg(path, route_file):
    path.write_text(f"""<?xml version="1.0" encoding="UTF-8"?>
<configuration>
    <input>
        <net-file value="intersection.net.xml"/>
        <route-files value="{route_file}"/>
        <additional-files value="vtypes.add.xml"/>
    </input>
    <time>
        <begin value="0"/>
        <step-length value="1"/>
    </time>
    <processing>
        <time-to-teleport value="-1"/>
    </processing>
    <report>
        <no-step-log value="true"/>
        <duration-log.disable value="true"/>
    </report>
</configuration>
""", encoding="utf-8")


def scaled_rate(flow, *, all_scale=1.0, ew_scale=1.0, ns_scale=1.0):
    origin = flow_origin(flow)
    axis_scale = ew_scale if origin in {"E", "W"} else ns_scale
    return float(flow.get("vehsPerHour")) * all_scale * axis_scale


def make_scaled_scenario(name, *, duration_s=1800, all_scale=1.0, ew_scale=1.0, ns_scale=1.0):
    src = Path("simulation/net/intersection.rou.xml")
    route_path = Path(f"simulation/net/{name}.rou.xml")
    sumocfg_path = Path(f"simulation/net/{name}.sumocfg")
    tree = ET.parse(src)
    root = tree.getroot()

    for flow in root.findall("flow"):
        flow.set("end", str(duration_s))
        flow.set("vehsPerHour", f"{scaled_rate(flow, all_scale=all_scale, ew_scale=ew_scale, ns_scale=ns_scale):.2f}")

    ET.indent(tree, space="    ")
    tree.write(route_path, encoding="UTF-8", xml_declaration=True)
    write_sumocfg(sumocfg_path, route_path.name)
    return str(sumocfg_path)


def make_time_block_scenario(name):
    """One episode that changes demand over time, so the final policy does
    not overfit to whichever single scenario was trained last."""
    src = Path("simulation/net/intersection.rou.xml")
    route_path = Path(f"simulation/net/{name}.rou.xml")
    sumocfg_path = Path(f"simulation/net/{name}.sumocfg")
    base_tree = ET.parse(src)
    base_root = base_tree.getroot()

    root = ET.Element(base_root.tag, base_root.attrib)
    for route in base_root.findall("route"):
        root.append(deepcopy(route))

    blocks = [
        ("normal", 0, 600, dict(all_scale=1.0)),
        ("heavy2x", 600, 1200, dict(all_scale=2.0)),
        ("ew3x", 1200, 1800, dict(ew_scale=3.0)),
        ("ns3x", 1800, 2400, dict(ns_scale=3.0)),
    ]
    for block_name, begin, end, multipliers in blocks:
        for flow in base_root.findall("flow"):
            block_flow = deepcopy(flow)
            block_flow.set("id", f"{flow.get('id')}_{block_name}")
            block_flow.set("begin", str(begin))
            block_flow.set("end", str(end))
            block_flow.set("vehsPerHour", f"{scaled_rate(flow, **multipliers):.2f}")
            root.append(block_flow)

    tree = ET.ElementTree(root)
    ET.indent(tree, space="    ")
    tree.write(route_path, encoding="UTF-8", xml_declaration=True)
    write_sumocfg(sumocfg_path, route_path.name)
    return str(sumocfg_path)


SCENARIOS = {
    "normal": "simulation/net/intersection.sumocfg",
    "heavy_2x": make_scaled_scenario("intersection_heavy_2x", all_scale=2.0),
    "imbalanced_ew3x": make_scaled_scenario("intersection_imbalanced_ew3x", ew_scale=3.0),
    "imbalanced_ns3x": make_scaled_scenario("intersection_imbalanced_ns3x", ns_scale=3.0),
    "mixed_balanced_peak": make_time_block_scenario("intersection_mixed_balanced_peak"),
}

SCENARIO_DURATIONS = {
    "normal": 1800,
    "heavy_2x": 1800,
    "imbalanced_ew3x": 1800,
    "imbalanced_ns3x": 1800,
    "mixed_balanced_peak": 2400,
}

SCENARIOS


## 3.1. Patch a balanced reward for Kaggle training

The project reward is intentionally simple: total waiting time delta. For the traffic-police goal, that is not enough because the agent can reduce total wait by starving a lighter road. This cell patches the Kaggle copy to add fairness pressure:

- keep the original total-wait delta reward;
- penalize total waiting level, worst phase waiting, phase imbalance, queue tail, active vehicles still in the network, and max individual vehicle wait;
- add a small moving-vehicle bonus so the policy is not rewarded only for reducing wait while sacrificing throughput;
- use a middle profile: throughput pressure is present but intentionally weaker than wait/queue pressure, because the previous strategy over-protected throughput and hurt heavy/EW waiting time;
- keep the guard stricter during training so the policy learns not to depend on very long red times.


In [ ]:
from pathlib import Path
import importlib

reward_path = Path("rl/reward/waiting_time_reward.py")
reward_path.write_text(r'''"""Balanced reward for traffic-police-style DQN training.

The base term keeps Sahal et al.'s total waiting-time delta:
    r_delta = previous_total_wait - current_total_wait

Additional terms discourage a policy that clears one axis while starving the
other. The weights are intentionally moderate: throughput still matters, but
very high phase waiting, queue tails, low movement, or individual waiting time becomes expensive.
"""

from __future__ import annotations

from common.constants import PHASE_DIRECTIONS, STOPPED_SPEED_THRESHOLD_MPS
from simulation.state.grid_encoder import APPROACH_EDGE_BY_DIRECTION
from simulation.state.waiting_time import total_waiting_time


class WaitingTimeReward:
    def __init__(
        self,
        *,
        total_wait_delta_weight: float = 1.0,
        total_wait_level_weight: float = 0.028,
        max_phase_wait_weight: float = 0.115,
        phase_imbalance_weight: float = 0.10,
        queue_weight: float = 2.35,
        max_queue_weight: float = 5.5,
        queue_imbalance_weight: float = 3.0,
        active_vehicle_weight: float = 0.25,
        moving_vehicle_bonus_weight: float = 0.25,
        max_vehicle_wait_weight: float = 0.07,
    ) -> None:
        self.total_wait_delta_weight = total_wait_delta_weight
        self.total_wait_level_weight = total_wait_level_weight
        self.max_phase_wait_weight = max_phase_wait_weight
        self.phase_imbalance_weight = phase_imbalance_weight
        self.queue_weight = queue_weight
        self.max_queue_weight = max_queue_weight
        self.queue_imbalance_weight = queue_imbalance_weight
        self.active_vehicle_weight = active_vehicle_weight
        self.moving_vehicle_bonus_weight = moving_vehicle_bonus_weight
        self.max_vehicle_wait_weight = max_vehicle_wait_weight
        self._previous_total: float = 0.0

    def reset(self, traci_conn) -> float:  # noqa: ANN001
        self._previous_total = total_waiting_time(traci_conn)
        return self._previous_total

    def step(self, traci_conn) -> float:  # noqa: ANN001
        current_total = total_waiting_time(traci_conn)
        delta_reward = self._previous_total - current_total
        phase_waits, phase_queues, max_vehicle_wait, active_vehicle_count, moving_vehicle_count = self._phase_metrics(traci_conn)

        wait_values = list(phase_waits.values())
        queue_values = list(phase_queues.values())
        max_phase_wait = max(wait_values) if wait_values else 0.0
        phase_imbalance = max(wait_values) - min(wait_values) if len(wait_values) >= 2 else 0.0
        total_queue = sum(queue_values)
        max_queue = max(queue_values) if queue_values else 0
        queue_imbalance = max(queue_values) - min(queue_values) if len(queue_values) >= 2 else 0

        reward = (
            self.total_wait_delta_weight * delta_reward
            - self.total_wait_level_weight * current_total
            - self.max_phase_wait_weight * max_phase_wait
            - self.phase_imbalance_weight * phase_imbalance
            - self.queue_weight * total_queue
            - self.max_queue_weight * max_queue
            - self.queue_imbalance_weight * queue_imbalance
            - self.active_vehicle_weight * active_vehicle_count
            + self.moving_vehicle_bonus_weight * moving_vehicle_count
            - self.max_vehicle_wait_weight * max_vehicle_wait
        )
        self._previous_total = current_total
        return reward

    @staticmethod
    def _phase_metrics(traci_conn):  # noqa: ANN001
        phase_edges = {
            phase.name: {APPROACH_EDGE_BY_DIRECTION[direction] for direction in directions}
            for phase, directions in PHASE_DIRECTIONS.items()
        }
        waits = {phase_name: 0.0 for phase_name in phase_edges}
        queues = {phase_name: 0 for phase_name in phase_edges}
        max_vehicle_wait = 0.0
        active_vehicle_count = 0
        moving_vehicle_count = 0

        for vehicle_id in traci_conn.vehicle.getIDList():
            road_id = traci_conn.vehicle.getRoadID(vehicle_id)
            waiting_time = traci_conn.vehicle.getWaitingTime(vehicle_id)
            speed = traci_conn.vehicle.getSpeed(vehicle_id)
            for phase_name, edges in phase_edges.items():
                if road_id in edges:
                    waits[phase_name] += waiting_time
                    active_vehicle_count += 1
                    if speed >= 1.0:
                        moving_vehicle_count += 1
                    if speed < STOPPED_SPEED_THRESHOLD_MPS:
                        queues[phase_name] += 1
                    max_vehicle_wait = max(max_vehicle_wait, waiting_time)
                    break

        return waits, queues, max_vehicle_wait, active_vehicle_count, moving_vehicle_count
''', encoding="utf-8")

import rl.reward.waiting_time_reward as reward_module
importlib.reload(reward_module)
from rl.reward.waiting_time_reward import WaitingTimeReward

assert hasattr(WaitingTimeReward(), "_phase_metrics")
BALANCED_REWARD_PATCHED = True
print("Balanced reward patched:", reward_path)


## 4. Quick pipeline smoke test (optional but recommended)

Runs 2 tiny episodes end-to-end before committing to a long training run - catches setup problems in seconds instead of hours.

In [ ]:
from rl.train.config import TrainingConfig
from rl.train.train import train

smoke_cfg = TrainingConfig(
    num_episodes=2,
    episode_duration_s=90,
    checkpoint_dir="/kaggle/working/smoke_checkpoints",
    min_replay_size=8,
    batch_size=4,
    max_red_time_s=None,
    soft_red_time_s=70.0,
    hard_red_time_s=120.0,
    starving_queue_threshold=4,
    starving_wait_time_s=160.0,
)
_ = train(smoke_cfg)
print("Smoke test OK")


## 5. Train the balanced traffic-police policy

This curriculum avoids the previous failure mode where the final model became good at one peak direction but regressed on normal traffic. It still uses staged training, but finishes on `mixed_balanced_peak`, where one episode contains normal, balanced-heavy, EW peak, and NS peak blocks.

Guard settings are stricter than the last run:

- `soft_red_time_s=70`
- `hard_red_time_s=120`
- `starving_queue_threshold=4`
- `starving_wait_time_s=160`

This does not mean every red must switch at 70s. It means after 70s the opposite phase must get service if it has real queue/waiting pressure, and 120s is the hard emergency cap.


In [ ]:
from pathlib import Path

from rl.train.config import TrainingConfig
from rl.train.train import train

CHECKPOINT_DIR = "/kaggle/working/checkpoints"

GUARD_CONFIG = dict(
    max_red_time_s=None,
    soft_red_time_s=70.0,
    hard_red_time_s=120.0,
    starving_queue_threshold=4,
    starving_wait_time_s=160.0,
)

TRAINING_BASE = dict(
    green_duration_s=10.0,
    checkpoint_dir=CHECKPOINT_DIR,
    batch_size=128,
    min_replay_size=2000,
    replay_capacity=100_000,
    learning_rate=5e-5,
    gamma=0.99,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay_episodes=900,
    target_sync_every_episodes=5,
    checkpoint_every_episodes=100,
    train_every_n_steps=4,
    log_every_episodes=10,
    **GUARD_CONFIG,
)

TRAIN_STAGE_DURATIONS = {
    "normal": 900,
    "heavy_2x": 1000,
    "mixed_balanced_peak": 1200,
    "imbalanced_ew3x": 1200,
    "imbalanced_ns3x": 1200,
    "mixed_balanced_peak_refresh": 1200,
    "imbalanced_ns3x_guardrail_refresh": 1200,
}


def train_stage(name, *, sumocfg_path, num_episodes, resume_from=None, episode_duration_s=None):
    duration = episode_duration_s or TRAIN_STAGE_DURATIONS.get(name) or SCENARIO_DURATIONS.get(name, 1800)
    print(f"\n=== Training stage: {name} -> episode {num_episodes}, duration={duration}s ===")
    cfg = TrainingConfig(
        sumocfg_path=sumocfg_path,
        num_episodes=num_episodes,
        episode_duration_s=duration,
        resume_from=resume_from,
        **TRAINING_BASE,
    )
    return train(cfg)


agent = train_stage("normal", sumocfg_path=SCENARIOS["normal"], num_episodes=250)
agent = train_stage(
    "heavy_2x",
    sumocfg_path=SCENARIOS["heavy_2x"],
    num_episodes=450,
    resume_from=f"{CHECKPOINT_DIR}/dqn_final.pt",
)
agent = train_stage(
    "mixed_balanced_peak",
    sumocfg_path=SCENARIOS["mixed_balanced_peak"],
    num_episodes=650,
    resume_from=f"{CHECKPOINT_DIR}/dqn_final.pt",
)
agent = train_stage(
    "imbalanced_ew3x",
    sumocfg_path=SCENARIOS["imbalanced_ew3x"],
    num_episodes=850,
    resume_from=f"{CHECKPOINT_DIR}/dqn_final.pt",
)
agent = train_stage(
    "imbalanced_ns3x",
    sumocfg_path=SCENARIOS["imbalanced_ns3x"],
    num_episodes=1050,
    resume_from=f"{CHECKPOINT_DIR}/dqn_final.pt",
)
agent = train_stage(
    "mixed_balanced_peak_refresh",
    sumocfg_path=SCENARIOS["mixed_balanced_peak"],
    num_episodes=1200,
    resume_from=f"{CHECKPOINT_DIR}/dqn_final.pt",
    episode_duration_s=TRAIN_STAGE_DURATIONS["mixed_balanced_peak_refresh"],
)
agent = train_stage(
    "imbalanced_ns3x_guardrail_refresh",
    sumocfg_path=SCENARIOS["imbalanced_ns3x"],
    num_episodes=1300,
    resume_from=f"{CHECKPOINT_DIR}/dqn_final.pt",
    episode_duration_s=TRAIN_STAGE_DURATIONS["imbalanced_ns3x_guardrail_refresh"],
)


## 6. Resuming (only needed if a session got cut off)

Kaggle sessions get killed after ~9-12 hours. `/kaggle/working/` output persists between sessions, so if training didn't finish, start a new session and continue from the last checkpoint instead of restarting from scratch.

In [ ]:
# from rl.train.config import TrainingConfig
# from rl.train.train import train
#
# resumed_cfg = TrainingConfig(
#     num_episodes=1400,
#     episode_duration_s=1200,
#     checkpoint_dir="/kaggle/working/checkpoints",
#     resume_from="/kaggle/working/checkpoints/dqn_final.pt",
# )
# agent = train(resumed_cfg)

## 7. Evaluate checkpoints and select the deployment model

This evaluates recent periodic checkpoints plus `dqn_best.pt` and `dqn_final.pt` on held-out seeds. The selected checkpoint is copied to `/kaggle/working/checkpoints/dqn_eval_best.pt`.

The score is intentionally stricter than before. It gives extra weight to normal/heavy/mixed scenarios and adds a penalty when DQN is worse than fixed-time on mean waiting, queue, or throughput.

Important: this cell also applies hard guardrails. A checkpoint that improves Heavy/EW but fails NS3x mean wait, final-wait tail, queue, or throughput receives a huge penalty and should not be selected as `dqn_eval_best.pt`. This prevents the v4 failure mode where Heavy/EW recovered but North-South collapsed.


In [ ]:
from pathlib import Path
import shutil

from benchmark.policies import DQNPolicy, FixedTimePolicy
from benchmark.run_episode import run_episode
from rl.agent.dqn_agent import DQNAgent
from rl.env.traffic_env import SumoTrafficEnv
from rl.train.checkpoint import load_checkpoint

CHECKPOINT_DIR = Path("/kaggle/working/checkpoints")
EVAL_SEEDS = [101, 102, 103]
EVAL_SCENARIOS = ["normal", "heavy_2x", "imbalanced_ew3x", "imbalanced_ns3x", "mixed_balanced_peak"]
EVAL_DURATIONS = {
    "normal": 900,
    "heavy_2x": 1200,
    "imbalanced_ew3x": 1200,
    "imbalanced_ns3x": 1200,
    "mixed_balanced_peak": 1600,
}
SCENARIO_WEIGHTS = {
    "normal": 1.35,
    "heavy_2x": 1.55,
    "imbalanced_ew3x": 1.15,
    "imbalanced_ns3x": 1.65,
    "mixed_balanced_peak": 1.70,
}

periodic = sorted(CHECKPOINT_DIR.glob("dqn_episode_*.pt"), key=lambda p: int(p.stem.split("_")[-1]))
candidate_paths = []
for path in [CHECKPOINT_DIR / "dqn_best.pt", CHECKPOINT_DIR / "dqn_final.pt", *periodic[-8:]]:
    if path.exists() and path not in candidate_paths:
        candidate_paths.append(path)


def mean(values):
    return sum(values) / len(values) if values else 0.0


def aggregate(metrics):
    dicts = [m.to_dict() for m in metrics]
    return {key: mean([d[key] for d in dicts]) for key in dicts[0]}


def score(metrics):
    # Lower is better. Max waiting matters because it reveals starvation even
    # when mean waiting looks acceptable.
    return (
        2.5 * metrics["mean_waiting_time_s"]
        + 1.5 * metrics["final_waiting_time_s"]
        + 1.2 * metrics["max_waiting_time_s"]
        + 52.0 * metrics["mean_queue_length"]
        + 30.0 * metrics["max_queue_length"]
        - 2.1 * metrics["arrived_vehicles"]
    )


def baseline_penalty(dqn_metrics, fixed_metrics):
    penalty = 0.0
    penalty += max(0.0, dqn_metrics["mean_waiting_time_s"] - 1.10 * fixed_metrics["mean_waiting_time_s"]) * 8.0
    penalty += max(0.0, dqn_metrics["mean_queue_length"] - 1.05 * fixed_metrics["mean_queue_length"]) * 120.0
    penalty += max(0.0, dqn_metrics["max_queue_length"] - 1.05 * fixed_metrics["max_queue_length"]) * 55.0
    penalty += max(0.0, dqn_metrics["final_waiting_time_s"] - 1.05 * fixed_metrics["final_waiting_time_s"]) * 3.0
    penalty += max(0.0, 0.990 * fixed_metrics["arrived_vehicles"] - dqn_metrics["arrived_vehicles"]) * 60.0
    return penalty


HARD_GUARDRAIL_PENALTY = 10_000_000.0


def improvement_pct(fixed_metrics, dqn_metrics):
    return {
        "mean_wait": (fixed_metrics["mean_waiting_time_s"] - dqn_metrics["mean_waiting_time_s"]) / fixed_metrics["mean_waiting_time_s"] * 100,
        "final_wait": (fixed_metrics["final_waiting_time_s"] - dqn_metrics["final_waiting_time_s"]) / fixed_metrics["final_waiting_time_s"] * 100 if fixed_metrics["final_waiting_time_s"] else 0.0,
        "queue": (fixed_metrics["mean_queue_length"] - dqn_metrics["mean_queue_length"]) / fixed_metrics["mean_queue_length"] * 100,
        "arrived": (dqn_metrics["arrived_vehicles"] - fixed_metrics["arrived_vehicles"]) / fixed_metrics["arrived_vehicles"] * 100,
    }


def hard_guardrail_failures(details):
    failures = []

    if details["normal"]["improvement"]["mean_wait"] < 20.0:
        failures.append("normal_mean_wait_regressed")
    if details["normal"]["improvement"]["arrived"] < 10.0:
        failures.append("normal_throughput_regressed")

    if details["heavy_2x"]["improvement"]["mean_wait"] < 5.0:
        failures.append("heavy_mean_wait_regressed")
    if details["heavy_2x"]["improvement"]["arrived"] < -8.0:
        failures.append("heavy_throughput_drop")
    if details["heavy_2x"]["improvement"]["queue"] < -8.0:
        failures.append("heavy_queue_regressed")

    if details["imbalanced_ew3x"]["improvement"]["mean_wait"] < 50.0:
        failures.append("ew3x_mean_wait_regressed")
    if details["imbalanced_ew3x"]["improvement"]["queue"] < 0.0:
        failures.append("ew3x_queue_regressed")
    if details["imbalanced_ew3x"]["improvement"]["arrived"] < 80.0:
        failures.append("ew3x_throughput_regressed")

    if details["imbalanced_ns3x"]["improvement"]["mean_wait"] < 10.0:
        failures.append("ns3x_mean_wait_regressed")
    if details["imbalanced_ns3x"]["improvement"]["final_wait"] < -40.0:
        failures.append("ns3x_final_wait_tail_regressed")
    if details["imbalanced_ns3x"]["improvement"]["queue"] < -10.0:
        failures.append("ns3x_queue_regressed")
    if details["imbalanced_ns3x"]["improvement"]["arrived"] < 5.0:
        failures.append("ns3x_throughput_regressed")

    if details["mixed_balanced_peak"]["improvement"]["mean_wait"] < 60.0:
        failures.append("mixed_mean_wait_regressed")
    if details["mixed_balanced_peak"]["improvement"]["queue"] < 0.0:
        failures.append("mixed_queue_regressed")
    if details["mixed_balanced_peak"]["improvement"]["arrived"] < 20.0:
        failures.append("mixed_throughput_regressed")

    return failures


def eval_policy(scenario_name, policy, seed):
    env = SumoTrafficEnv(
        sumocfg_path=SCENARIOS[scenario_name],
        episode_duration_s=EVAL_DURATIONS[scenario_name],
        green_duration_s=10.0,
        backend="libsumo",
        **GUARD_CONFIG,
    )
    try:
        return run_episode(env, policy, seed=seed)
    finally:
        env.close()


fixed_by_scenario = {}
print("Fixed-time baselines:")
for scenario_name in EVAL_SCENARIOS:
    fixed_metrics = [
        eval_policy(scenario_name, FixedTimePolicy(green_duration_s=20.0), seed)
        for seed in EVAL_SEEDS
    ]
    agg = aggregate(fixed_metrics)
    fixed_by_scenario[scenario_name] = agg
    print(scenario_name, agg, "score=", score(agg))

results = []
guardrail_failures_by_path = {}
for path in candidate_paths:
    total_score = 0.0
    details = {}
    trained_episode = None
    for scenario_name in EVAL_SCENARIOS:
        agent = DQNAgent()
        trained_episode = load_checkpoint(path, agent)
        policy = DQNPolicy(agent, epsilon=0.0)
        metrics = [eval_policy(scenario_name, policy, seed) for seed in EVAL_SEEDS]
        agg = aggregate(metrics)
        fixed = fixed_by_scenario[scenario_name]
        improvement = improvement_pct(fixed, agg)
        scenario_score = score(agg) + baseline_penalty(agg, fixed)
        weighted_score = SCENARIO_WEIGHTS[scenario_name] * scenario_score
        total_score += weighted_score
        details[scenario_name] = {
            "metrics": agg,
            "raw_score": score(agg),
            "baseline_penalty": baseline_penalty(agg, fixed),
            "improvement": improvement,
            "weighted_score": weighted_score,
        }
    failures = hard_guardrail_failures(details)
    guardrail_failures_by_path[path.name] = failures
    if failures:
        total_score += HARD_GUARDRAIL_PENALTY * len(failures)
    results.append((total_score, path, trained_episode, details))
    print(f"\n{path.name:<22} ep={trained_episode:<5} total_score={total_score:10.2f}")
    if failures:
        print("  HARD_GUARDRAIL_FAIL", failures)
    for scenario_name, detail in details.items():
        agg = detail["metrics"]
        fixed = fixed_by_scenario[scenario_name]
        mean_wait_improvement = (fixed["mean_waiting_time_s"] - agg["mean_waiting_time_s"]) / fixed["mean_waiting_time_s"] * 100
        print(
            f"  {scenario_name:<20} mean_wait={agg['mean_waiting_time_s']:8.2f} "
            f"impr={mean_wait_improvement:7.1f}% queue={agg['mean_queue_length']:6.2f} "
            f"arrived={agg['arrived_vehicles']:7.2f} penalty={detail['baseline_penalty']:8.1f}"
        )

results.sort(key=lambda row: row[0])
best_score, best_path, best_episode, best_details = results[0]
selected_path = CHECKPOINT_DIR / "dqn_eval_best.pt"
shutil.copy2(best_path, selected_path)
print("\nSelected:", best_path.name, "episode", best_episode, "score", best_score)
print("Guardrail failures:", guardrail_failures_by_path.get(best_path.name, []))
print("Wrote:", selected_path)


## 8. Download the selected model

Download `dqn_eval_best.pt` first. It is selected by deterministic evaluation across held-out seeds, so it is usually a better deployment candidate than the raw training `dqn_best.pt`. Keep `dqn_best.pt` and `dqn_final.pt` too if you want to compare locally.

In [ ]:
!ls -lh /kaggle/working/checkpoints


In [ ]:
import base64
from pathlib import Path
from IPython.display import HTML, display


def download_link(path, filename=None):
    filename = filename or path.split("/")[-1]
    with open(path, "rb") as f:
        data = f.read()
    b64 = base64.b64encode(data).decode()
    return HTML(f'<a download="{filename}" href="data:application/octet-stream;base64,{b64}">Download {filename}</a>')


for path in [
    "/kaggle/working/checkpoints/dqn_eval_best.pt",
    "/kaggle/working/checkpoints/dqn_best.pt",
    "/kaggle/working/checkpoints/dqn_final.pt",
]:
    if Path(path).exists():
        display(download_link(path))
